In [2]:
import os
import time
import mne
import numpy as np
import pandas as pd

import bsl
from bsl import StreamPlayer, datasets
# from bsl.externals import pylsl  # distributed version of pylsl
from bsl.triggers import TriggerDef

import pylsl

import pickle

import math
import matplotlib
import matplotlib.pyplot as plt
from pythonosc.udp_client import SimpleUDPClient

In [5]:
from pylsl import resolve_streams

streams = resolve_streams()
for s in streams:
    print(s.name(), s.type(), s.channel_count(), s.nominal_srate())

OxySoft NIRS 28 75.0
OxySoft Event Marker Markers 1 0.0


### OSC Client Intialization

In [3]:
# OSC client initialization
ip = "127.0.0.1"
port = 4545
client = SimpleUDPClient(ip, port)

### Load Pretrained Model

In [4]:
# Load your pretrained model and scaler
model = pickle.load("model.pkl")
scaler = pickle.load("scaler.pkl")

TypeError: file must have 'read' and 'readline' attributes

## Analyse Signal

In [ ]:
#TODO: Might want to change the buffer size and window size
receiver = bsl.StreamReceiver(bufsize=10, winsize=10, stream_name=['eeg', 'fnirs_device_name'])

#TODO: this channels will need to be updated 
eeg_info = mne.create_info(sfreq=250, ch_names=['E1'], ch_types=['eeg'])
fnirs_info = mne.create_info(sfreq=10, ch_names=['HbO', 'HbR'], ch_types=['fnirs_cw_amplitude'])
#TODO: some preprocessing here with baseline correction and baseline alpha power to normalize scores 
ref_mean_score = 0 
ref_std_score = 0
while True:
    receiver.acquire()

    #TODO: change stream name
    eeg_data, _ = receiver.get_window(stream_name='igeb')
    eeg_data = np.nan_to_num(eeg_data)
    # EEG: drop timestamp, keep EEG channel
    eeg_raw = mne.io.RawArray(data=eeg_data[:, [False, True]].T, info=eeg_info)
    eeg_raw.filter(1, 30)
    eeg_raw.crop(tmin=9)
    psds, _ = mne.time_frequency.psd_welch(eeg_raw, fmin=8, fmax=12, n_fft=125)
    alpha_score = np.mean(psds)
    alpha_norm = (alpha_score - ref_mean_score) / ref_std_score / 2

    
    #TODO: change stream name
    fnirs_data, fnirs_ts = receiver.get_window(stream_name='fnirs_device')

    # fNIRS: drop timestamp, keep oxy/deoxy channels
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T, info=fnirs_info)
    fnirs_data = np.nan_to_num(fnirs_data)
    # Assuming channel 0 = timestamp, 1 = HbO, 2 = HbR
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T,
                                 info=fnirs_info)
    fnirs_raw.filter(0.01, 0.5)  # Hemodynamic response is slow
    fnirs_raw.crop(tmin=9)
    # Convert to concentration (requires MNE fnirs processing)
    # fnirs_conc = mne.preprocessing.nirs.beer_lambert_law(fnirs_raw)
    # Or just use raw amplitude features
    hbo_mean = fnirs_raw.get_data(picks=['HbO']).mean()
    hbr_mean = fnirs_raw.get_data(picks=['HbR']).mean()

    features = np.array([[alpha_norm, hbo_mean, hbr_mean]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)

In [ ]:
from pylsl import resolve_streams, StreamInlet
from collections import deque
import numpy as np
from scipy.signal import butter, sosfilt
import mne
import time

# ── Constants ────────────────────────────────────────────────────────────────
EEG_FS = 250
FNIRS_FS = 75.0
WINDOW_S = 10
EEG_WINDOW_SAMPLES = int(EEG_FS * WINDOW_S)    # 2500 samples
FNIRS_WINDOW_SAMPLES = int(FNIRS_FS * WINDOW_S) # 750 samples
UPDATE_EVERY_S = 0.5
N_FNIRS_CHANNELS = 28

HBO_COLS = list(range(0, N_FNIRS_CHANNELS, 2))
HBR_COLS = list(range(1, N_FNIRS_CHANNELS, 2))

# ── MNE info objects (TODO: update with real values from stream) ──────────────
eeg_info = mne.create_info(sfreq=EEG_FS, ch_names=['E1'], ch_types=['eeg'])

ch_names = [
    'Rx1 - Tx1 O2Hb', 'Rx1 - Tx1 HHb',
    'Rx1 - Tx3 O2Hb', 'Rx1 - Tx3 HHb',
    'Rx2 - Tx1 O2Hb', 'Rx2 - Tx1 HHb',
    'Rx2 - Tx3 O2Hb', 'Rx2 - Tx3 HHb',
    'Rx3 - Tx4 O2Hb', 'Rx3 - Tx4 HHb',
    'Rx3 - Tx5 O2Hb', 'Rx3 - Tx5 HHb',
    'Rx8 - Tx9 O2Hb', 'Rx8 - Tx9 HHb',
    'Rx8 - Tx10 O2Hb', 'Rx8 - Tx10 HHb',
    'Rx5 - Tx6 O2Hb', 'Rx5 - Tx6 HHb',
    'Rx5 - Tx8 O2Hb', 'Rx5 - Tx8 HHb',
    'Rx6 - Tx6 O2Hb', 'Rx6 - Tx6 HHb',
    'Rx6 - Tx8 O2Hb', 'Rx6 - Tx8 HHb',
    'Rx4 - Tx2 O2Hb', 'Rx4 - Tx2 HHb',
    'Rx7 - Tx7 O2Hb', 'Rx7 - Tx7 HHb',
]
fnirs_info = mne.create_info(ch_names=ch_names, sfreq=FNIRS_FS, ch_types=['fnirs_cw_amplitude'] * 28)

# ── Baseline normalization (TODO: compute from real baseline recording) ───────
ref_mean_score = 0
ref_std_score = 1  # avoid div by zero

# ── EEG bandpass filter 1-30Hz ───────────────────────────────────────────────
def make_bandpass(lowcut, highcut, fs, order=4):
    nyq = fs / 2
    sos = butter(order, [lowcut / nyq, highcut / nyq], btype='band', output='sos')
    return sos

eeg_sos = make_bandpass(1, 30, EEG_FS)

# ── LSL inlets ───────────────────────────────────────────────────────────────
streams = resolve_streams()
eeg_stream = [s for s in streams if s.type() == 'EEG'][0]
fnirs_stream = [s for s in streams if s.name() == 'OxySoft'][0]
eeg_inlet = StreamInlet(eeg_stream)
fnirs_inlet = StreamInlet(fnirs_stream)

# ── Ring buffers ─────────────────────────────────────────────────────────────
eeg_buffer = deque(maxlen=EEG_WINDOW_SAMPLES)
fnirs_buffer = deque(maxlen=FNIRS_WINDOW_SAMPLES)

# ── Main loop ─────────────────────────────────────────────────────────────────
last_process_time = time.time()

while True:
    # Ingest both streams
    eeg_samples, _ = eeg_inlet.pull_chunk(timeout=0.1)
    if eeg_samples:
        eeg_buffer.extend(eeg_samples)

    fnirs_samples, _ = fnirs_inlet.pull_chunk(timeout=0.0)  # non-blocking second call
    if fnirs_samples:
        fnirs_buffer.extend(fnirs_samples)

    # Wait for both buffers to fill on startup
    if len(eeg_buffer) < EEG_WINDOW_SAMPLES or len(fnirs_buffer) < FNIRS_WINDOW_SAMPLES:
        print(f"Buffering... EEG {len(eeg_buffer)}/{EEG_WINDOW_SAMPLES} | fNIRS {len(fnirs_buffer)}/{FNIRS_WINDOW_SAMPLES}")
        continue

    # Rate limit processing
    now = time.time()
    if now - last_process_time < UPDATE_EVERY_S:
        continue
    last_process_time = now

    # ── EEG features ─────────────────────────────────────────────────────────
    eeg_data = np.array(eeg_buffer)             # (2500, n_eeg_channels)
    eeg_data = np.nan_to_num(eeg_data)

    # TODO: update column selection once real channel layout is known
    eeg_channel = eeg_data[:, 0]                # (2500,) single channel for now

    eeg_filt = sosfilt(eeg_sos, eeg_channel)    # bandpass 1-30Hz

    # Alpha power (8-12Hz) via Welch
    from scipy.signal import welch
    freqs, psd = welch(eeg_filt, fs=EEG_FS, nperseg=EEG_FS // 2)
    alpha_mask = (freqs >= 8) & (freqs <= 12)
    alpha_score = psd[alpha_mask].mean()
    alpha_norm = (alpha_score - ref_mean_score) / ref_std_score

    # ── fNIRS features ────────────────────────────────────────────────────────
    fnirs_data = np.array(fnirs_buffer)         # (750, 28)
    fnirs_data = np.nan_to_num(fnirs_data)

    hbo = fnirs_data[:, HBO_COLS]               # (750, 14)
    hbr = fnirs_data[:, HBR_COLS]               # (750, 14)

    hbo_mean = hbo.mean(axis=0)                 # (14,)
    hbr_mean = hbr.mean(axis=0)                 # (14,)

    t = np.arange(FNIRS_WINDOW_SAMPLES) / FNIRS_FS
    hbo_slope = np.polyfit(t, hbo, 1)[0]       # (14,)
    hbr_slope = np.polyfit(t, hbr, 1)[0]       # (14,)

    # ── Fusion & prediction ───────────────────────────────────────────────────
    features = np.concatenate([[alpha_norm], hbo_mean, hbr_mean, hbo_slope, hbr_slope])  # (57,)
    features_scaled = scaler.transform(features.reshape(1, -1))
    prediction = model.predict(features_scaled)

    print("Prediction:", prediction)

## Send Signal Via Osc

In [ ]:

client.send_message("/in/prediction", float(prediction[0]))

## open code code

Key design decisions:
- Window alignment: Both use the same 10s window, so features are temporally aligned
- Feature vector: [alpha_power, hbo_mean, hbr_mean] — you'd likely expand this (multiple EEG frequency bands, HbO/HbR slope, etc.)
- Scaler: fNIRS units (µM) differ wildly from EEG alpha power, so StandardScaler fit offline is essential
- Rate: The combined loop runs at whatever rate you set (4 Hz like the original, or slower to match fNIRS)

In [ ]:
# Main loop
while True:
    receiver.acquire()

    # --- EEG stream ---
    eeg_data, _ = receiver.get_window(stream_name='igeb')
    eeg_data = np.nan_to_num(eeg_data)
    eeg_raw = mne.io.RawArray(data=eeg_data[:, [False, True]].T, info=eeg_info)
    eeg_raw.filter(1, 30)
    eeg_raw.crop(tmin=9)
    psds, _ = mne.time_frequency.psd_welch(eeg_raw, fmin=8, fmax=12, n_fft=125)
    alpha_score = np.mean(psds)
    alpha_norm = (alpha_score - ref_mean_score) / ref_std_score / 2

    # --- fNIRS stream ---
    fnirs_data, _ = receiver.get_window(stream_name='fnirs_stream_name')
    fnirs_data = np.nan_to_num(fnirs_data)
    # Assuming channel 0 = timestamp, 1 = HbO, 2 = HbR
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T,
                                 info=fnirs_info)
    fnirs_raw.filter(0.01, 0.5)  # Hemodynamic response is slow
    fnirs_raw.crop(tmin=9)
    # Convert to concentration (requires MNE fnirs processing)
    # fnirs_conc = mne.preprocessing.nirs.beer_lambert_law(fnirs_raw)
    # Or just use raw amplitude features
    hbo_mean = fnirs_raw.get_data(picks=['HbO']).mean()
    hbr_mean = fnirs_raw.get_data(picks=['HbR']).mean()

    # --- Combine and predict ---
    features = np.array([[alpha_norm, hbo_mean, hbr_mean]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)

    # Send to VR/neuromore
    client.send_message("/in/prediction", float(prediction[0]))